In [20]:
# ── Chargement des 4 tables nettoyées ───────────────────────────

import pandas as pd
import numpy as np
from s3_utils import read_s3_csv, upload_to_s3

# On définit les chemins dans S3 (dossier silver)
# Note : Vérifie bien que les noms correspondent à ce que tu as envoyé précédemment
usagers   = read_s3_csv('silver/usagers_cleaned.csv', separator=',')
caract    = read_s3_csv('silver/caracteristiques_cleaned.csv', separator=',')
lieux     = read_s3_csv('silver/lieux_cleaned.csv', separator=',')
vehicules = read_s3_csv('silver/vehicules_cleaned.csv', separator=',')

# On force le type string sur Num_Acc pour être sûr que les jointures se fassent bien
# (Parfois les zéros au début sautent s'ils sont lus comme des nombres)
usagers['Num_Acc']   = usagers['Num_Acc'].astype(str)
caract['Num_Acc']    = caract['Num_Acc'].astype(str)
lieux['Num_Acc']     = lieux['Num_Acc'].astype(str)
vehicules['Num_Acc'] = vehicules['Num_Acc'].astype(str)

# Baptiste Dimanche NETTOYAGE RADICAL DES CLÉS
def force_string_clean(df, col):
    # On transforme en texte, on vire les .0 si c'est du float, et on retire les espaces
    return df[col].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

for d in [usagers, caract, lieux, vehicules]:
    d['Num_Acc'] = force_string_clean(d, 'Num_Acc')

usagers['id_vehicule'] = force_string_clean(usagers, 'id_vehicule')
vehicules['id_vehicule'] = force_string_clean(vehicules, 'id_vehicule')

print("✅ Clés nettoyées et harmonisées !")

# Baptiste : correction du type d'id_veihucle qui bloquait le merge
usagers['id_vehicule'] = usagers['id_vehicule'].astype(str)
vehicules['id_vehicule'] = vehicules['id_vehicule'].astype(str)

print(f"usagers   : {usagers.shape}")
print(f"caract    : {caract.shape}")
print(f"lieux     : {lieux.shape}")
print(f"vehicules : {vehicules.shape}")

c:\Users\bsacu\Desktop\FullStack\Projet final\Python\projet-groupe-jedha\s3_utils.py:23: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(io.BytesIO(response['Body'].read()), sep=separator)


✅ Clés nettoyées et harmonisées !
usagers   : (506886, 14)
caract    : (221044, 11)
lieux     : (252928, 6)
vehicules : (378071, 7)


In [21]:
# ── Vérification des clés de jointure ───────────────────────────

print("--- Types de Num_Acc ---")
for name, df in [("usagers", usagers), ("caract", caract),
                 ("lieux", lieux), ("vehicules", vehicules)]:
    print(f"  {name:<12} : {df['Num_Acc'].dtype} | NaN : {df['Num_Acc'].isna().sum()}")

print("\n--- Nb accidents uniques par table ---")
for name, df in [("usagers", usagers), ("caract", caract),
                 ("lieux", lieux), ("vehicules", vehicules)]:
    print(f"  {name:<12} : {df['Num_Acc'].nunique():,} accidents uniques")

--- Types de Num_Acc ---
  usagers      : object | NaN : 0
  caract       : object | NaN : 0
  lieux        : object | NaN : 0
  vehicules    : object | NaN : 0

--- Nb accidents uniques par table ---
  usagers      : 221,044 accidents uniques
  caract       : 165,743 accidents uniques
  lieux        : 221,044 accidents uniques
  vehicules    : 221,044 accidents uniques


In [22]:
# ── Jointure des 4 tables ───────────────────────────────────────

df = usagers.merge(caract, on='Num_Acc', how='left', suffixes=('', '_caract'))
print(f"Après merge caract    : {df.shape}")

# Dédoublonner lieux AVANT la jointure
lieux_dedup = lieux.drop_duplicates(subset=['Num_Acc'], keep='first')
print(f"lieux avant dédup : {lieux.shape[0]:,}")
print(f"lieux après dédup : {lieux_dedup.shape[0]:,}")

df = df.merge(lieux_dedup, on='Num_Acc', how='left', suffixes=('', '_lieux'))

df = df.merge(vehicules, on=['Num_Acc', 'id_vehicule'], how='left', suffixes=('', '_vehicules'))
print(f"Après merge vehicules : {df.shape}")

print(f"\n✅ Table finale : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")

Après merge caract    : (506886, 24)
lieux avant dédup : 252,928
lieux après dédup : 221,044
Après merge vehicules : (506886, 34)

✅ Table finale : 506,886 lignes x 34 colonnes


In [18]:
df.isna().sum()

Num_Acc                   0
id_usager                 0
id_vehicule               0
num_veh                   0
catu                      0
grav                    419
sexe                  10632
an_nais               30811
trajet                11269
secu1                  9865
secu2                223921
secu3                489711
locp                 240813
actp                      0
lum                  126662
departement          126662
commune              126662
agglomeration        126662
intersection         126662
meteo                126684
type_collision       126743
lat                  126662
long                 126662
date                 126662
catr                      0
nbv                    3816
surf                      0
infra                     0
vma                       0
num_veh_vehicules    506886
catv                 506886
obs                  506886
obsm                 506886
motor                506886
dtype: int64

In [7]:
# ── Nettoyage post-jointure ─────────────────────────────────────

cols_to_drop = [col for col in df.columns if col.endswith('_caract')
                                           or col.endswith('_lieux')
                                           or col.endswith('_vehicules')]
df = df.drop(columns=cols_to_drop)

print(f"✅ Colonnes dupliquées supprimées")
print(f"Shape final : {df.shape}")
print(f"Colonnes : {list(df.columns)}")

✅ Colonnes dupliquées supprimées
Shape final : (506886, 33)
Colonnes : ['Num_Acc', 'id_usager', 'id_vehicule', 'num_veh', 'catu', 'grav', 'sexe', 'an_nais', 'trajet', 'secu1', 'secu2', 'secu3', 'locp', 'actp', 'lum', 'departement', 'commune', 'agglomeration', 'intersection', 'meteo', 'type_collision', 'lat', 'long', 'date', 'catr', 'nbv', 'surf', 'infra', 'vma', 'catv', 'obs', 'obsm', 'motor']


In [8]:
# ── Vérification finale ─────────────────────────────────────────

print(f"Shape final : {df.shape}")

print("\n--- NaN par colonne ---")
nan_df = pd.DataFrame({
    'NaN count': df.isna().sum(),
    'NaN %': (df.isna().mean() * 100).round(1)
})
print(nan_df[nan_df['NaN count'] > 0])

print(f"\n--- Lignes perdues vs usagers ---")
print(f"Usagers avant jointure : {len(usagers):,}")
print(f"Table finale           : {len(df):,}")
print(f"Différence             : {len(usagers) - len(df):,}")

Shape final : (506886, 33)

--- NaN par colonne ---
                NaN count  NaN %
grav                  419    0.1
sexe                10632    2.1
an_nais             30811    6.1
trajet              11269    2.2
secu1                9865    1.9
secu2              223921   44.2
secu3              489711   96.6
locp               240813   47.5
lum                506886  100.0
departement        506886  100.0
commune            506886  100.0
agglomeration      506886  100.0
intersection       506886  100.0
meteo              506886  100.0
type_collision     506886  100.0
lat                506886  100.0
long               506886  100.0
date               506886  100.0
nbv                  3816    0.8
catv               506886  100.0
obs                506886  100.0
obsm               506886  100.0
motor              506886  100.0

--- Lignes perdues vs usagers ---
Usagers avant jointure : 506,886
Table finale           : 506,886
Différence             : 0


In [ ]:
# ── CELLULE 6 : Export de la table finale ─────────────────────────────────── mis en stase pour la suite.

"""df.to_csv('Dataset/dataset_complet_clean.csv', index=False, sep=';', encoding='utf-8')
print(f"✅ Exporté — {len(df):,} lignes x {df.shape[1]} colonnes")"""

✅ Exporté — 506,886 lignes x 38 colonnes


In [ ]:
# Exportation du fichier sur Gold
file_name = "master_accidents_data.csv"

# Envoi vers le dossier GOLD
print("🚀 Envoi de la table Master vers S3 (GOLD)...")
upload_to_s3(df, file_name, folder="gold")

print(f"✅ Mission accomplie ! Le fichier est dans 'gold/{file_name}'")